In [19]:
import pandas as pd
import re
import html
import time
import json
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from tqdm import tqdm

In [20]:

ALGOLIA_URL = "https://hn.algolia.com/api/v1/search_by_date"

from datetime import datetime, timezone, timedelta
import time
import requests

def fetch_hn(
    keyword: str,
    tag: str = "story",
    hits_per_page: int = 100,
    max_pages: int = 4,
    sleep_s: float = 0.2,
    days: int | None = None,
):
    rows = []

    # compute cutoff once
    cutoff_ts = None
    if days is not None:
        cutoff_ts = int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp())

    for page in range(max_pages):
        params = {
            "query": keyword,
            "tags": tag,
            "hitsPerPage": hits_per_page,
            "page": page,
        }

        # add time filter only if requested
        if cutoff_ts is not None:
            params["numericFilters"] = [f"created_at_i>={cutoff_ts}"]

        r = requests.get(ALGOLIA_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        for hit in data.get("hits", []):
            rows.append({
                "source": "HackerNews",
                "keyword": keyword,
                "content_type": tag,
                "objectID": hit.get("objectID"),
                "created_at": hit.get("created_at"),
                "author": hit.get("author"),
                "url": hit.get("url"),
                "story_title": hit.get("title") or hit.get("story_title"),
                "text": (hit.get("comment_text") or hit.get("story_text") or ""),
            })

        # stop if last available page
        if page >= data.get("nbPages", 1) - 1:
            break

        if sleep_s:
            time.sleep(sleep_s)

    return rows


# --- Example usage ---
if __name__ == "__main__":
    kw = "payments"
    stories = fetch_hn(kw, tag="story", max_pages=4)
    comments = fetch_hn(kw, tag="comment", max_pages=4)
    print("stories:", len(stories), "comments:", len(comments))

stories: 400 comments: 400


In [21]:


KEYWORDS_TO_TRACK = ["film", "platform", "creator", "monetization"]

all_rows = []

for kw in tqdm(KEYWORDS_TO_TRACK, desc="Fetching HN"):
    story_rows = fetch_hn(kw, tag="story", max_pages=5, days=7)
    comment_rows = fetch_hn(kw, tag="comment", max_pages=5, days=7)

    print(f"{kw}: stories={len(story_rows)} comments={len(comment_rows)}")

    all_rows.extend(story_rows)
    all_rows.extend(comment_rows)

df_raw = pd.DataFrame(all_rows)

# ✅ create a stable dedupe key (objectID alone can clash across content types)
df_raw["dedupe_id"] = (
    df_raw["content_type"].astype(str).fillna("unknown") + ":" +
    df_raw["objectID"].astype(str).fillna("missing")
)

# ✅ drop duplicates safely
df_raw = df_raw.drop_duplicates(subset=["dedupe_id"]).copy()

# ✅ metadata
df_raw["fetched_at"] = datetime.now(timezone.utc).isoformat()

# Optional: keep track of which keyword produced the row (already in df_raw["keyword"])
# Optional: sort by created_at (if present)
# df_raw = df_raw.sort_values("created_at", ascending=False)

print("Rows after dedupe:", len(df_raw))
df_raw.shape, df_raw.columns

Fetching HN:  25%|█████████████████████████████████████████▎                                                                                                                           | 1/4 [00:13<00:40, 13.62s/it]

film: stories=462 comments=500


Fetching HN:  50%|██████████████████████████████████████████████████████████████████████████████████▌                                                                                  | 2/4 [00:24<00:24, 12.02s/it]

platform: stories=203 comments=500


Fetching HN:  75%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                         | 3/4 [00:30<00:09,  9.16s/it]

creator: stories=69 comments=500


Fetching HN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:31<00:00,  7.80s/it]

monetization: stories=4 comments=39
Rows after dedupe: 2155


((2155, 11),
 Index(['source', 'keyword', 'content_type', 'objectID', 'created_at', 'author',
        'url', 'story_title', 'text', 'dedupe_id', 'fetched_at'],
       dtype='object'))

In [22]:

def clean_text_fn(s: str) -> str:
    if s is None:
        return ""
    s = str(s)

    # Decode HTML entities: &gt; &quot; &#x2F; etc.
    s = html.unescape(s)

    # Remove HTML tags like <p> <pre> <code> <a href=...>
    s = re.sub(r"<[^>]+>", " ", s)

    # Remove leftover escaped artifacts
    s = s.replace("\\n", " ").replace("\\t", " ")

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [23]:
df = df_raw.copy()

# ✅ Combine title + body so short stories don't get dropped
df["clean_text"] = (
    df["story_title"].fillna("") + ". " + df["text"].fillna("")
).apply(clean_text_fn)

df["clean_length"] = df["clean_text"].str.len()

# Drop empty and very short texts
df = df[df["clean_length"] >= 40].copy()

print("Rows after length filter:", len(df))

df.shape, df[["text", "clean_text"]].head(3)

Rows after length filter: 2132


((2132, 13),
                                                 text  \
 0  smplogs analyzes your AWS CloudWatch log expor...   
 1  Hey HN,<p>For the past few weekends, I have be...   
 2  KARTA is a command center where hiring teams z...   
 
                                           clean_text  
 0  Show HN: Smplogs – Local-first AWS Cloudwatch ...  
 1  Show HN: Turning 2D floor plans into 3D-ready ...  
 2  Show HN: Karta – Google Search, for discoverin...  )

In [25]:
LUNIM_KEYWORDS = [
    "market gap", "customer need", "user need",
    "pain point", "problem", "friction",
    "platform", "tool", "product", "solution", "alternative",
    "use case", "value", "benefit", "advantage",
    "adoption", "workflow", "efficiency", "automation",
    "pricing", "cost", "revenue", "monetization"
]

NOISE_KEYWORDS = [
    # force job stuff to False
    "who is hiring", "hiring", "job", "role", "position", "resume", "cv",
    # common HN noise
    "kernel", "compiler", "filesystem", "mmap", "rust", "c++", "linux",
    "crypto", "token", "blockchain"
]

def mark_lunim_relevance(text: str) -> bool:
    if not isinstance(text, str):
        return False
    t = text.lower()

    # hard reject noise first
    if any(nk in t for nk in NOISE_KEYWORDS):
        return False

    # accept if any market keyword appears
    return any(lk in t for lk in LUNIM_KEYWORDS)

In [28]:
df["is_relevant"] = df["clean_text"].apply(mark_lunim_relevance)

print(df["is_relevant"].value_counts())
df[df["is_relevant"] == True][["clean_text"]].head(5)

is_relevant
False    1258
True      874
Name: count, dtype: int64


,clean_text
5,"Show HN: EloPhanto – Video creation, 116 tools..."
12,Show HN: I built an open-source analytics plat...
14,Show HN: OnGarde – Runtime content security pr...
19,Show HN: Portable Secret – Open-Source HTML fi...
22,Show HN: BetterDB Cloud – monitor Valkey/Redis...


In [29]:
import pandas as pd
from datetime import datetime, timezone, timedelta

dt = pd.to_datetime(df_relevant["created_at"], utc=True, errors="coerce")
cutoff = datetime.now(timezone.utc) - timedelta(days=7)

print("Relevant total:", len(df_relevant))
print("Relevant within 7 days:", (dt >= cutoff).sum())
print("Newest relevant:", dt.max())

Relevant total: 868
Relevant within 7 days: 868
Newest relevant: 2026-02-26 18:44:19+00:00


In [30]:
df_relevant = df[df["is_relevant"] == True].copy()
df_noise    = df[df["is_relevant"] == False].copy()

df_relevant.shape, df_noise.shape

((874, 14), (1258, 14))

In [31]:
from datetime import datetime, timezone
from pathlib import Path

# Robust project root detection (manual + automation)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "frontend":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

# Timestamped saves
df.to_csv(DATA_DIR / f"hn_market_labeled_{ts}.csv", index=False)
df_relevant.to_csv(DATA_DIR / f"hn_market_relevant_{ts}.csv", index=False)
df_noise.to_csv(DATA_DIR / f"hn_market_noise_{ts}.csv", index=False)

# Latest always-overwritten versions
df.to_csv(DATA_DIR / "hn_market_labeled_latest.csv", index=False)
df_relevant.to_csv(DATA_DIR / "hn_market_relevant_latest.csv", index=False)
df_noise.to_csv(DATA_DIR / "hn_market_noise_latest.csv", index=False)

print("Saved files with timestamp:", ts)
print("Latest relevant:", DATA_DIR / "hn_market_relevant_latest.csv")

Saved files with timestamp: 20260226_1856
Latest relevant: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_market_relevant_latest.csv
